# 🚀 AI YouTube Studio — Colab GPU Worker [NODE 3 — CHẠY SONG SONG]
### Trạm Render GPU NVIDIA T4/A100 Siêu Tốc (20-30s/Video) — Tự Động Quét & Hoàn Thiện Video Từ Telegram & Google Drive!

---
### 📌 HƯỚNG DÙNG 1-CHẠM TRÊN ĐIỆN THOẠI HOẶC MÁY TÍNH:
1. **Bấm Chạy Tất Cả:** Vào menu **Runtime** (Thời gian chạy) ➡️ Bấm **Run all** (Chạy tất cả).
2. **Cấp quyền Drive:** Bấm **Connect to Google Drive** (khi hiện cửa sổ hỏi).
3. **TẮT MÀN HÌNH HOẶC THOÁT LINK ĐƯỢC NGAY:**
   - Colab sẽ tự động quét toàn bộ kịch bản do Bot Telegram gửi lên Drive (`/AI_YouTube_RenderFarm/pending/`).
   - Tự động hỏi và xử lý nốt các video dở dang (nếu có từ phiên trước).
   - Sinh giọng đọc Edge-TTS + khớp phụ đề WhisperX GPU + tải stock B-Roll 9:16 + render NVENC chỉ trong 20-30 giây.
   - Video hoàn thành tự động lưu vào `/AI_YouTube_RenderFarm/completed/` để Bot lấy về gửi cho bạn!


In [ ]:
# [BƯỚC 1] Kiểm tra sức mạnh cạc đồ họa GPU & Hạn mức phiên làm việc
import subprocess

print('=' * 65)
print('🚀 AI YOUTUBE STUDIO — KIỂM TRA HẠN MỨC GPU WORKER')
print('=' * 65)

try:
    gpu_info = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total,utilization.gpu', '--format=csv,noheader']
    ).decode('utf-8').strip()
    print(f'✅ GPU ĐANG KẾT NỐI: {gpu_info}')
    print('✨ Tình trạng: Sẵn sàng render video siêu tốc 20-30s/video!')
    print('💡 Lưu ý: Colab chỉ dùng GPU máy ảo dựng video, KHÔNG tốn hạn mức Gemini Ultra.')
except Exception:
    print('⚠️ CẢNH BÁO: Phiên Colab này chưa có cạc GPU (đang chạy bằng CPU)!')
    print('   👉 1. Vào menu: Runtime (Thời gian chạy) ➡️ Change runtime type ➡️ Chọn T4 GPU.')
    print('   👉 2. Nếu Google báo hết hạn mức GPU hôm nay: Hãy bấm sang Link Tài Khoản 2 hoặc 3!')

print('=' * 65)
!nvidia-smi


In [ ]:
# [BƯỚC 2] Cài đặt các thư viện siêu tốc cho GPU (WhisperX, Edge-TTS, FFmpeg NVENC)
!pip install -q edge-tts faster-whisper pillow requests pydantic python-dotenv
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
print("✅ Đã cài đặt xong môi trường GPU Worker Siêu Tốc!")

In [ ]:
# [BƯỚC 3] Kết nối Google Drive & Thiết lập Không Gian Render Farm (Hỗ trợ xoay 3 Tài Khoản)
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"ℹ️ Lỗi kết nối Google Drive: {e}")

# Tự động phát hiện thư mục Render Farm trên MyDrive hoặc Shared with me
base_drive = Path('/content/drive')
candidate = base_drive / 'MyDrive' / 'AI_YouTube_RenderFarm'

if not candidate.exists():
    # Quét tìm nếu được tạo dưới dạng lối tắt hoặc thư mục chia sẻ
    found = list(base_drive.glob('**/AI_YouTube_RenderFarm'))
    if found:
        candidate = found[0]

DRIVE_FARM_DIR = candidate
PENDING_DIR = DRIVE_FARM_DIR / 'pending'
IN_PROGRESS_DIR = DRIVE_FARM_DIR / 'in_progress'
COMPLETED_DIR = DRIVE_FARM_DIR / 'completed'
ARCHIVE_DIR = DRIVE_FARM_DIR / 'archive'

if DRIVE_FARM_DIR.exists() and (candidate / 'pending').exists():
    for d in [PENDING_DIR, IN_PROGRESS_DIR, COMPLETED_DIR, ARCHIVE_DIR]:
        d.mkdir(parents=True, exist_ok=True)
    print('✅ ĐÃ KẾT NỐI VÀO KHO DỮ LIỆU RENDER FARM DÙNG CHUNG!')
    print(f'   📁 Thư mục làm việc:                 {DRIVE_FARM_DIR}')
    print(f'   📁 Hàng đợi chờ render (pending):     {PENDING_DIR}')
    print(f'   📁 Video đang dựng (in_progress):   {IN_PROGRESS_DIR}')
    print(f'   📁 Video hoàn thành (completed):     {COMPLETED_DIR}')
else:
    print('=' * 70)
    print('🚨 CẢNH BÁO: CHƯA KẾT NỐI VÀO THƯ MỤC AI_YouTube_RenderFarm CHUNG!')
    print('=' * 70)
    print('👉 NẾU ĐÂY LÀ TÀI KHOẢN PHỤ (TK 2 HOẶC TK 3):')
    print('   1. Mở https://drive.google.com bằng tài khoản Google này.')
    print('   2. Vào mục Được chia sẻ với tôi (Shared with me) ở cột bên trái.')
    print('   3. Chuột phải vào thư mục AI_YouTube_RenderFarm ➡️ Chọn Thêm lối tắt vào Drive (Add shortcut to Drive) ➡️ Chọn Drive của tôi (My Drive).')
    print('   4. Bấm chạy lại Bước 3 này là xong!')
    print('👉 NẾU ĐÂY LÀ TÀI KHOẢN CÁ NHÂN KHÁC (NGOÀI CỤM 3 TÀI KHOẢN):')
    print('   Vui lòng đổi sang 1 trong 3 tài khoản hệ thống:')
    print('   • TK 1: pool1.studio@example.com')
    print('   • TK 2: pool2.studio@example.com')
    print('   • TK 3: pool3.studio@example.com')
    print('=' * 70)


In [ ]:
# [BƯỚC 4] Bộ Máy Dựng Video GPU Toàn Năng (Full Render Engine)
# Tích hợp VideoStrategistAgent (Anti-Slideshow): Phân biệt Video chuyển động thực, Tư liệu lịch sử & AI Viễn tưởng
import os
import re
import json
import time
import shutil
import asyncio
import urllib.parse
import requests
import subprocess
from pathlib import Path
from PIL import Image

def get_gpu_encoder():
    """Kiểm tra xem hệ thống có hỗ trợ bộ mã hóa phần cứng h264_nvenc không"""
    try:
        res = subprocess.run(["ffmpeg", "-encoders"], capture_output=True, text=True)
        if "h264_nvenc" in res.stdout:
            return "h264_nvenc"
    except Exception:
        pass
    return "libx264"

ENCODER = get_gpu_encoder()
print(f"🎮 Bộ mã hóa Video đang sử dụng: {ENCODER} ({'GPU Tăng Tốc 20s' if ENCODER == 'h264_nvenc' else 'CPU'})")

async def generate_voice_edge_tts(text: str, voice: str, output_path: str) -> bool:
    """Sinh giọng đọc tự nhiên bằng Edge-TTS trong 1-2 giây"""
    import edge_tts
    clean_text = re.sub(r'[*_#`~]', '', text).strip()
    if not clean_text:
        return False
    communicate = edge_tts.Communicate(clean_text, voice)
    await communicate.save(output_path)
    return Path(output_path).exists() and Path(output_path).stat().st_size > 1000

def transcribe_with_whisper(audio_path: str, language: str = None):
    """Dùng Faster-Whisper trên CUDA GPU để lấy timestamp chuẩn từng từ trong 2s, khóa ngôn ngữ chuẩn xác"""
    try:
        from faster_whisper import WhisperModel
        model = WhisperModel("small", device="cuda", compute_type="float16")
        transcribe_kwargs = {"word_timestamps": True}
        if language:
            transcribe_kwargs["language"] = language
        segments, info = model.transcribe(audio_path, **transcribe_kwargs)
        words = []
        for seg in segments:
            for w in seg.words:
                words.append({"word": w.word.strip(), "start": w.start, "end": w.end})
        return words
    except Exception as e:
        print(f"   ⚠️ WhisperX GPU fallback: {e}")
        return []

def create_dynamic_ass_subtitles(words: list, output_path: str, width: int = 1080, height: int = 1920, total_duration: float = 30.0, is_vietnamese: bool = False):
    """Tạo file phụ đề hoạt họa ASS chuyên nghiệp chuẩn Shorts dọc kèm CTA Banner kích thích đăng ký kênh"""
    header = f"""[Script Info]
ScriptType: v4.00+
PlayResX: {width}
PlayResY: {height}
ScaledBorderAndShadow: yes

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,Arial,66,&H00FFFFFF,&H000000FF,&H00000000,&H80000000,-1,0,0,0,100,100,0,0,1,5,2,2,40,40,480,1
Style: Highlight,Arial,70,&H0000FFFF,&H000000FF,&H00000000,&H80000000,-1,0,0,0,105,105,0,0,1,6,3,2,40,40,480,1
Style: CTABadge,Arial,46,&H0000FFFF,&H000000FF,&H00000000,&H80000000,-1,0,0,0,100,100,0,0,1,5,3,2,40,40,750,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
    def fmt_time(sec):
        h = int(sec // 3600)
        m = int((sec % 3600) // 60)
        s = int(sec % 60)
        cs = int(round((sec - int(sec)) * 100))
        return f"{h:01d}:{m:02d}:{s:02d}.{cs:02d}"

    events = []
    chunk_size = 4
    for i in range(0, len(words), chunk_size):
        chunk = words[i:i+chunk_size]
        if not chunk:
            continue
        start_t = fmt_time(chunk[0]["start"])
        end_t = fmt_time(chunk[-1]["end"] + 0.15)
        line_text = " ".join([w["word"] for w in chunk])
        events.append(f"Dialogue: 0,{start_t},{end_t},Default,,0,0,0,,{line_text}")

    # ── Hiển thị Animated CTA Banner ở 4.5 giây cuối video ──
    if total_duration > 4.0:
        cta_start = fmt_time(max(0.0, total_duration - 4.5))
        cta_end = fmt_time(total_duration)
        cta_text = "🔔 ĐĂNG KÝ KÊNH ĐÓN XEM PHẦN 2 🔔" if is_vietnamese else "🔔 SUBSCRIBE FOR PART 2 🔔"
        events.append(f"Dialogue: 1,{cta_start},{cta_end},CTABadge,,0,0,0,,{{\\fade(250,150)\\t(0,250,\\fscx110\\fscy110)\\t(250,500,\\fscx100\\fscy100)}}{cta_text}")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(header + "\n".join(events) + "\n")
    return output_path

def apply_ken_burns_to_image(img_path: str, out_path: str, duration: float, scene_id: int = 1):
    """Áp dụng chuyển động điện ảnh Ken Burns zoom/pan 1080x1920 30fps bằng GPU NVENC lên ảnh thật"""
    total_frames = max(30, int(duration * 30))
    if scene_id % 2 == 1:
        vf_zoom = f"scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,zoompan=z='min(zoom+0.0018,1.25)':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':d={total_frames}:s=1080x1920:fps=30,setsar=1"
    else:
        vf_zoom = f"scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,zoompan=z='if(lte(zoom,1.0),1.25,max(1.0,zoom-0.0018))':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':d={total_frames}:s=1080x1920:fps=30,setsar=1"
    enc_opts = [
        "-rc:v", "vbr", "-cq:v", "18", "-b:v", "12M",
        "-maxrate:v", "18M", "-bufsize:v", "24M",
        "-spatial_aq", "1", "-temporal_aq", "1", "-preset", "p5"
    ] if ENCODER == "h264_nvenc" else ["-crf", "18", "-preset", "medium"]
    cmd = [
        "ffmpeg", "-y",
        "-loop", "1", "-i", img_path,
        "-vf", vf_zoom,
        "-t", str(duration),
        "-c:v", ENCODER,
    ] + enc_opts + [
        "-pix_fmt", "yuv420p",
        "-an",
        out_path
    ]
    subprocess.run(cmd, check=True)
    return True

def search_and_download_scene_media(query: str, visual_prompt: str, out_path: str, duration: float, pexels_key: str = None, pixabay_key: str = None, scene_id: int = 1, scene_meta: dict = None):
    """
    Quy trình phân giải tài nguyên thị giác theo chiến lược thông minh của VideoStrategistAgent (Anti-Slideshow Manifesto):
    1. STOCK_FOOTAGE : Tìm video B-roll thực tế từ Pexels Video -> Pixabay Video -> Photo Ken Burns
    2. REAL_ARCHIVE  : Tìm tư liệu/hình ảnh thật lịch sử, đời sống thực tế (ảnh tư liệu xác thực)
    3. TEXT_TO_VIDEO : Viễn tưởng, siêu thực, trừu tượng -> Tìm Pixabay CGI 3D Video hoặc gọi Pollinations FLUX Engine
    4. IMAGE_TO_VIDEO: Tạo ảnh chi tiết nghệ thuật 8K bằng FLUX -> Ken Burns motion
    """
    meta = scene_meta or {}
    prod_method = meta.get("production_method") or meta.get("visual_strategy") or "STOCK_FOOTAGE"
    sq = meta.get("search_query") or query or ""
    clean_q = " ".join(re.sub(r'[^a-zA-Z0-9 ]', '', sq).split()[:4])
    if not clean_q:
        clean_q = "cinematic motion"
    ai_p = meta.get("ai_prompt") or visual_prompt or f"cinematic 9:16 vertical shot of {clean_q}, hyper-detailed, 8k, dramatic lighting"

    # Helper 1: Tải Pexels Video (Ưu tiên số 1 cho video chuyển động thực tế)
    def try_pexels_video():
        if not pexels_key:
            return False
        try:
            headers = {"Authorization": pexels_key}
            url = f"https://api.pexels.com/videos/search?query={urllib.parse.quote(clean_q)}&orientation=portrait&per_page=3"
            res = requests.get(url, headers=headers, timeout=8)
            if res.status_code == 200:
                vids = res.json().get("videos", [])
                for v in vids:
                    files = v.get("video_files", [])
                    portrait = [f for f in files if f.get("width", 0) <= f.get("height", 0) and "mp4" in f.get("file_type", "mp4")]
                    if portrait:
                        ideal = sorted([f for f in portrait if f.get("height", 0) >= 1080], key=lambda x: x.get("height", 0), reverse=True)
                        if not ideal:
                            ideal = sorted([f for f in portrait if f.get("height", 0) >= 720], key=lambda x: x.get("height", 0), reverse=True)
                        chosen = ideal[0] if ideal else portrait[0]
                        link = chosen.get("link")
                        if link:
                            temp_raw = f"/tmp/raw_px_{time.time()}.mp4"
                            r_down = requests.get(link, timeout=25)
                            if r_down.status_code == 200 and len(r_down.content) > 5000:
                                with open(temp_raw, "wb") as f_raw:
                                    f_raw.write(r_down.content)
                                enc_opts = [
                                    "-rc:v", "vbr", "-cq:v", "18", "-b:v", "12M",
                                    "-maxrate:v", "18M", "-bufsize:v", "24M",
                                    "-spatial_aq", "1", "-temporal_aq", "1", "-preset", "p5"
                                ] if ENCODER == "h264_nvenc" else ["-crf", "18", "-preset", "medium"]
                                cmd = [
                                    "ffmpeg", "-y",
                                    "-stream_loop", "-1",
                                    "-i", temp_raw,
                                    "-t", str(duration),
                                    "-vf", "scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,setsar=1,fps=30",
                                    "-c:v", ENCODER,
                                ] + enc_opts + [
                                    "-pix_fmt", "yuv420p",
                                    "-an",
                                    out_path
                                ]
                                subprocess.run(cmd, check=True)
                                Path(temp_raw).unlink(missing_ok=True)
                                print(f"   🎬 [Pexels Video] Đã tải clip chuyển động thật Cảnh {scene_id} ('{clean_q}')")
                                return True
        except Exception:
            pass
        return False

    # Helper 2: Tải Pixabay Video (hỗ trợ CGI 3D / Sci-fi / B-Roll)
    def try_pixabay_video(search_term):
        if not pixabay_key:
            return False
        try:
            url_pix = f"https://pixabay.com/api/videos/?key={pixabay_key}&q={urllib.parse.quote(search_term)}&per_page=3"
            res_pix = requests.get(url_pix, timeout=8)
            if res_pix.status_code == 200:
                hits = res_pix.json().get("hits", [])
                for h in hits:
                    v_files = h.get("videos", {})
                    v_info = v_files.get("large") or v_files.get("medium") or v_files.get("small")
                    v_url = v_info.get("url") if v_info else None
                    if v_url:
                        temp_raw = f"/tmp/raw_pix_{time.time()}.mp4"
                        r_down = requests.get(v_url, timeout=25)
                        if r_down.status_code == 200 and len(r_down.content) > 5000:
                            with open(temp_raw, "wb") as f_raw:
                                f_raw.write(r_down.content)
                            cmd = [
                                "ffmpeg", "-y",
                                "-stream_loop", "-1",
                                "-i", temp_raw,
                                "-t", str(duration),
                                "-vf", "scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,setsar=1,fps=30",
                                "-c:v", ENCODER,
                                "-preset", "fast",
                                "-pix_fmt", "yuv420p",
                                "-an",
                                out_path
                            ]
                            subprocess.run(cmd, check=True)
                            Path(temp_raw).unlink(missing_ok=True)
                            print(f"   🎬 [Pixabay Video] Đã tải clip cho Cảnh {scene_id} ('{search_term}')")
                            return True
        except Exception:
            pass
        return False

    # Helper 3: Tải Ảnh tư liệu thực tế (Pexels / Pixabay) -> Áp dụng Ken Burns Motion
    def try_real_photo():
        if pexels_key:
            try:
                headers = {"Authorization": pexels_key}
                url = f"https://api.pexels.com/v1/search?query={urllib.parse.quote(clean_q)}&orientation=portrait&per_page=3"
                res = requests.get(url, headers=headers, timeout=8)
                if res.status_code == 200:
                    photos = res.json().get("photos", [])
                    if photos:
                        img_url = photos[0].get("src", {}).get("large2x") or photos[0].get("src", {}).get("large")
                        if img_url:
                            temp_img = f"/tmp/px_img_{time.time()}.jpg"
                            r_img = requests.get(img_url, timeout=20)
                            if r_img.status_code == 200 and len(r_img.content) > 5000:
                                with open(temp_img, "wb") as fi:
                                    fi.write(r_img.content)
                                apply_ken_burns_to_image(temp_img, out_path, duration, scene_id)
                                Path(temp_img).unlink(missing_ok=True)
                                print(f"   📸 [Tư liệu thật Pexels] Ken Burns Cảnh {scene_id} ('{clean_q}')")
                                return True
            except Exception:
                pass
        if pixabay_key:
            try:
                url_pix = f"https://pixabay.com/api/?key={pixabay_key}&q={urllib.parse.quote(clean_q)}&image_type=photo&orientation=vertical&per_page=3"
                res_pix = requests.get(url_pix, timeout=8)
                if res_pix.status_code == 200:
                    hits = res_pix.json().get("hits", [])
                    if hits:
                        img_url = hits[0].get("largeImageURL") or hits[0].get("webformatURL")
                        if img_url:
                            temp_img = f"/tmp/pix_img_{time.time()}.jpg"
                            r_img = requests.get(img_url, timeout=20)
                            if r_img.status_code == 200 and len(r_img.content) > 5000:
                                with open(temp_img, "wb") as fi:
                                    fi.write(r_img.content)
                                apply_ken_burns_to_image(temp_img, out_path, duration, scene_id)
                                Path(temp_img).unlink(missing_ok=True)
                                print(f"   📸 [Tư liệu thật Pixabay] Ken Burns Cảnh {scene_id} ('{clean_q}')")
                                return True
            except Exception:
                pass
        return False

    # Helper 4: Sinh cảnh siêu thực AI FLUX (Pollinations AI 9:16)
    def try_flux_ai(prompt_text):
        try:
            clean_prompt = " ".join(prompt_text.split()[:50])
            encoded = urllib.parse.quote(clean_prompt)
            seed = int(time.time() * 1000) % 999999 + scene_id
            flux_url = f"https://image.pollinations.ai/prompt/{encoded}?width=768&height=1344&model=flux&nologo=true&seed={seed}"
            temp_img = f"/tmp/flux_{time.time()}_{scene_id}.jpg"
            r_flux = requests.get(flux_url, timeout=20)
            if r_flux.status_code == 200 and len(r_flux.content) > 5000:
                with open(temp_img, "wb") as ff:
                    ff.write(r_flux.content)
                apply_ken_burns_to_image(temp_img, out_path, duration, scene_id)
                Path(temp_img).unlink(missing_ok=True)
                print(f"   🎨 [FLUX AI Generation] Sinh cảnh nghệ thuật AI Cảnh {scene_id} ({prod_method})!")
                return True
        except Exception as e_flux:
            print(f"   ⚠️ Lỗi FLUX: {e_flux}")
        return False

    # ── ĐIỀU HƯỚNG THEO ĐÚNG CHIẾN LƯỢC VIDEO STRATEGIST ──
    # Trường hợp 1: Viễn tưởng / Huyền bí / Siêu thực (TEXT_TO_VIDEO / AI_VIDEO)
    if prod_method in ("TEXT_TO_VIDEO", "AI_VIDEO"):
        print(f"   🔮 [Chiến lược: VIỄN TƯỞNG / AI] Cảnh {scene_id}: Ưu tiên Video CGI 3D hoặc Sinh AI...")
        if try_pixabay_video(f"3d sci-fi {clean_q}") or try_pixabay_video("sci-fi abstract"):
            return True
        if try_flux_ai(ai_p):
            return True
        if try_pexels_video():
            return True

    # Trường hợp 2: Thứ có thật / Lịch sử / Bằng chứng (REAL_ARCHIVE / REAL_DOCUMENTARY)
    elif prod_method in ("REAL_ARCHIVE", "REAL_DOCUMENTARY"):
        print(f"   🏛️ [Chiến lược: TƯ LIỆU THỰC TẾ] Cảnh {scene_id}: Trích xuất ảnh/video dẫn chứng có thật...")
        if try_real_photo():
            return True
        if try_pexels_video():
            return True
        if try_flux_ai(ai_p):
            return True

    # Trường hợp 3: Ảnh nghệ thuật chi tiết (IMAGE_TO_VIDEO / AI_IMAGE)
    elif prod_method in ("IMAGE_TO_VIDEO", "AI_IMAGE"):
        print(f"   🎨 [Chiến lược: ẢNH NGHỆ THUẬT AI] Cảnh {scene_id}: Sinh ảnh 8K chi tiết & Motion...")
        if try_flux_ai(ai_p):
            return True
        if try_real_photo():
            return True

    # Trường hợp 4 mặc định: Video chuyển động thực tế (STOCK_FOOTAGE / STOCK_VIDEO)
    else:
        print(f"   🎬 [Chiến lược: VIDEO CHUYỂN ĐỘNG THỰC] Cảnh {scene_id}: Tìm clip B-Roll 9:16...")
        if try_pexels_video():
            return True
        if try_pixabay_video(clean_q):
            return True
        if try_real_photo():
            return True
        if try_flux_ai(ai_p):
            return True

    # Fallback cứu nguy cuối cùng nếu từ khóa quá khó hoặc mạng chập chờn: Thử lại FLUX với từ khóa đơn giản
    if try_flux_ai(f"cinematic {clean_q} 8k highly detailed vertical wallpaper"):
        return True

    # Fallback an toàn nếu mọi nguồn mạng đều lỗi: Tạo nền gradient động thay vì khối đen đơn điệu
    temp_img = f"/tmp/bg_{scene_id}.jpg"
    colors = ["#0f172a", "#1e1b4b", "#1e293b", "#111827", "#172554", "#14532d", "#3b0764"]
    c_pick = colors[scene_id % len(colors)]
    cmd_img = ["ffmpeg", "-y", "-f", "lavfi", "-i", f"color=c={c_pick}:s=1080x1920:d=1", "-frames:v", "1", temp_img]
    subprocess.run(cmd_img, check=True)
    apply_ken_burns_to_image(temp_img, out_path, duration, scene_id)
    Path(temp_img).unlink(missing_ok=True)
    return True


In [ ]:
# [BƯỚC 5] TỰ ĐỘNG PHÁT HIỆN TÁC VỤ KỊCH BẢN & SẴN SÀNG SẢN XUẤT (100% TỰ ĐỘNG - KHÔNG CẦN NHẬP LỆNH)
import time
import sys
import json
from pathlib import Path

print("=" * 75)
print("🔍 TỰ ĐỘNG QUÉT CÁC TÁC VỤ KỊCH BẢN TỪ BOT TELEGRAM TRÊN GOOGLE DRIVE...")
print("=" * 75)

# Đảm bảo đường dẫn Drive tồn tại
if "DRIVE_FARM_DIR" not in globals() or not DRIVE_FARM_DIR.exists():
    base_drive = Path('/content/drive')
    candidate = base_drive / 'MyDrive' / 'AI_YouTube_RenderFarm'
    if not candidate.exists():
        found = list(base_drive.glob('**/AI_YouTube_RenderFarm'))
        candidate = found[0] if found else candidate
    DRIVE_FARM_DIR = candidate

BASE_DRIVE_DIR = DRIVE_FARM_DIR
PENDING_DIR = DRIVE_FARM_DIR / 'pending'
IN_PROGRESS_DIR = DRIVE_FARM_DIR / 'in_progress'
COMPLETED_DIR = DRIVE_FARM_DIR / 'completed'
ARCHIVE_DIR = DRIVE_FARM_DIR / 'archive'
LOCKS_DIR = DRIVE_FARM_DIR / 'locks'
COORDINATOR_DIR = DRIVE_FARM_DIR / 'coordinator'

for d in [PENDING_DIR, IN_PROGRESS_DIR, COMPLETED_DIR, ARCHIVE_DIR, LOCKS_DIR, COORDINATOR_DIR]:
    d.mkdir(parents=True, exist_ok=True)

pending_tasks = sorted(list(PENDING_DIR.glob("task_*.json")))
in_progress_tasks = sorted(list(IN_PROGRESS_DIR.glob("task_*.json")))

print(f"📊 KẾT QUẢ QUÉT HÀNG ĐỢI:")
print(f"   • 📥 Video MỚI đang chờ:       {len(pending_tasks)} video")
for t in pending_tasks:
    try:
        d = json.load(open(t, encoding='utf-8'))
        print(f"     └─ [Job #{d.get('job_id')}] {d.get('title', 'Không có tiêu đề')} ({len(d.get('scenes', []))} cảnh)")
    except Exception:
        print(f"     └─ {t.name}")

if in_progress_tasks:
    print(f"   • ⚠️ Video DỞ DANG từ phiên trước: {len(in_progress_tasks)} video")
    for t in in_progress_tasks:
        try:
            d = json.load(open(t, encoding='utf-8'))
            print(f"     └─ [Job #{d.get('job_id')}] {d.get('title', 'Không có tiêu đề')} (Bị ngắt giữa chừng)")
        except Exception:
            print(f"     └─ {t.name}")

# TỰ ĐỘNG GỘP TẤT CẢ (Ưu tiên video dở dang chạy trước, sau đó đến video mới) - 100% ZERO INPUT
raw_tasks = in_progress_tasks + pending_tasks
seen_jobs = set()
tasks_to_run = []
for t in raw_tasks:
    try:
        d = json.load(open(t, encoding='utf-8'))
        jid = d.get('job_id')
        if jid not in seen_jobs:
            seen_jobs.add(jid)
            tasks_to_run.append(t)
    except Exception:
        tasks_to_run.append(t)

if tasks_to_run:
    print("-" * 75)
    print(f"🚀 TỰ ĐỘNG SẢN XUẤT {len(tasks_to_run)} VIDEO BẰNG GPU NVENC!")
    print("👉 Bạn không cần nhập bất kỳ phím nào! CÓ THỂ ĐÓNG TAB HOẶC TẮT MÀN HÌNH NGAY.")
    print("👉 Khi render xong, Colab sẽ tự ngắt kết nối để tiết kiệm 100% Compute Units!")
    print("-" * 75)
else:
    print("-" * 75)
    print("☕ Hàng đợi hiện tại trống (0 video). Không có video nào cần xử lý.")
    print("👉 Colab sẽ tự động ngắt kết nối để tiết kiệm tài nguyên.")
    print("-" * 75)


In [ ]:
# [BƯỚC 6] THỰC THI SẢN XUẤT VIDEO SIÊU TỐC BẰNG GPU & ĐẨY VỀ DRIVE (HỖ TRỢ ĐA WORKER SONG SONG)
import time
import os
import shutil
import subprocess
import json
from pathlib import Path

print("=" * 75)
print("🚀 KHỞI ĐỘNG HỆ THỐNG COLAB GPU RENDER FARM (MULTI-WORKER)")
print("=" * 75)

# Đảm bảo BASE_DRIVE_DIR và các thư mục luôn được định nghĩa đầy đủ
if "DRIVE_FARM_DIR" in globals() and DRIVE_FARM_DIR.exists():
    BASE_DRIVE_DIR = DRIVE_FARM_DIR
else:
    candidate = Path("/content/drive/MyDrive/AI_YouTube_RenderFarm")
    if not candidate.exists():
        found = list(Path("/content/drive").glob("**/AI_YouTube_RenderFarm"))
        candidate = found[0] if found else candidate
    BASE_DRIVE_DIR = candidate

DRIVE_FARM_DIR = BASE_DRIVE_DIR
PENDING_DIR = BASE_DRIVE_DIR / "pending"
IN_PROGRESS_DIR = BASE_DRIVE_DIR / "in_progress"
COMPLETED_DIR = BASE_DRIVE_DIR / "completed"
ARCHIVE_DIR = BASE_DRIVE_DIR / "archive"
LOCKS_DIR = BASE_DRIVE_DIR / "locks"
COORDINATOR_DIR = BASE_DRIVE_DIR / "coordinator"

for p in [PENDING_DIR, IN_PROGRESS_DIR, COMPLETED_DIR, ARCHIVE_DIR, LOCKS_DIR, COORDINATOR_DIR]:
    p.mkdir(parents=True, exist_ok=True)

STATE_FILE = COORDINATOR_DIR / "cluster_state.json"
NODE_NUMBER = 3
WORKER_NODE_NAME = "Colab-Worker-Node3"
print(f"💻 Đang hoạt động với danh tính: {WORKER_NODE_NAME} (Node {NODE_NUMBER})")

def update_cluster_heartbeat(status="RUNNING", current_job=None, error=None):
    try:
        st = json.load(open(STATE_FILE, encoding='utf-8')) if STATE_FILE.exists() else {}
        nodes = st.get("cluster_nodes", {})
        nkey = str(NODE_NUMBER)
        if nkey in nodes:
            nodes[nkey]["status"] = status
            nodes[nkey]["last_heartbeat"] = time.strftime("%Y-%m-%dT%H:%M:%SZ")
            nodes[nkey]["current_job_id"] = current_job
            if error:
                nodes[nkey]["error_message"] = error
            if status == "COMPLETED":
                nodes[nkey]["completed_count"] = nodes[nkey].get("completed_count", 0) + 1
        st["cluster_nodes"] = nodes
        with open(STATE_FILE, "w", encoding='utf-8') as sf:
            json.dump(st, sf, ensure_ascii=False, indent=2)
    except Exception as ex_st:
        print(f"⚠️ [Coordinator] Lỗi cập nhật trạng thái: {ex_st}")

def mark_quota_exhausted_and_rotate():
    try:
        st = json.load(open(STATE_FILE, encoding='utf-8')) if STATE_FILE.exists() else {}
        nodes = st.get("cluster_nodes", {})
        nkey = str(NODE_NUMBER)
        if nkey in nodes:
            nodes[nkey]["status"] = "QUOTA_EXHAUSTED"
        next_node = (NODE_NUMBER % 3) + 1
        st["active_node"] = next_node
        next_info = nodes.get(str(next_node), {})
        st["active_account"] = next_info.get("account", "pool1.studio@example.com")
        st["action_required"] = {
            "needs_user_run": True,
            "target_node": next_node,
            "target_node_name": next_info.get("name", f"Node {next_node}"),
            "target_account": st["active_account"],
            "direct_colab_url": f"https://accounts.google.com/AccountChooser?Email={st['active_account']}&continue=https%3A%2F%2Fcolab.research.google.com%2Fdrive%2F{next_info.get('notebook_id')}",
            "reason": f"Node {NODE_NUMBER} hết Compute Units/GPU -> Tự động xoay sang Node {next_node}!"
        }
        with open(STATE_FILE, "w", encoding='utf-8') as sf:
            json.dump(st, sf, ensure_ascii=False, indent=2)
        print(f"🚨 [QUOTA SHIFT] Đã chuyển quyền ưu tiên sang Node {next_node} ({st['active_account']})!")
    except Exception as ex_rot:
        print(f"⚠️ [Coordinator] Lỗi xoay vòng node: {ex_rot}")

# Lấy danh sách tác vụ từ Bước 5 hoặc tự quét nếu chạy cell lẻ
candidate_tasks = globals().get("tasks_to_run", [])
if not candidate_tasks:
    raw_tasks = sorted(list(IN_PROGRESS_DIR.glob("task_*.json"))) + sorted(list(PENDING_DIR.glob("task_*.json")))
    seen_j = set()
    candidate_tasks = []
    for t in raw_tasks:
        try:
            d = json.load(open(t, encoding='utf-8'))
            jid = d.get('job_id')
            if jid not in seen_j:
                seen_j.add(jid)
                candidate_tasks.append(t)
        except Exception:
            candidate_tasks.append(t)

active_tasks = []
for tf in candidate_tasks:
    if not tf.exists():
        continue
    lock_file = LOCKS_DIR / f"{tf.stem}.lock"
    if lock_file.exists():
        try:
            if time.time() - lock_file.stat().st_mtime > 1800:
                lock_file.unlink(missing_ok=True)
            else:
                continue
        except Exception:
            continue
    try:
        with open(lock_file, "w", encoding="utf-8") as f_l:
            f_l.write(f"{WORKER_NODE_NAME} | {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        active_tasks.append(tf)
    except Exception:
        pass

if not active_tasks:
    print("💤 Không có tác vụ mới nào cần dựng (hoặc các Node khác đang xử lý).")
else:
    print(f"🎯 Tìm thấy {len(active_tasks)} tác vụ sẽ được render trên {WORKER_NODE_NAME}!")
    update_cluster_heartbeat(status="RUNNING")


async def process_single_task(task_path: Path):
    t_start = time.time()
    with open(task_path, "r", encoding="utf-8") as f:
        task = json.load(f)

    job_id = task.get("job_id")
    title = task.get("title") or f"Video #{job_id}"
    voice = task.get("voice", "en-US-AndrewNeural")
    full_narration = task.get("full_narration", "")
    scenes = task.get("scenes", [])
    task_lang = task.get("language", "en")
    pexels_key = task.get("pexels_api_key")
    pixabay_key = task.get("pixabay_api_key")
    
    # Tự động điều chỉnh giọng đọc Edge-TTS tương thích ngôn ngữ
    is_vi = task_lang == "vi" or any(c in full_narration.lower() for c in "àáảãạăắằẳẵặâấầẩẫậèéẻẽẹêếềểễệìíỉĩịòóỏõọôốồổỗộơớờởỡợùúủũụưứừửữựỳýỷỹỵđ")
    if is_vi and not voice.startswith("vi-"):
        voice = "vi-VN-NamMinhNeural"
    elif not is_vi and voice.startswith("vi-"):
        voice = "en-US-AndrewNeural"

    print("-" * 70)
    print(f"🎬 BẮT ĐẦU SẢN XUẤT VIDEO #{job_id}: {title}")
    print(f"🗣️ Giọng đọc: {voice} | Ngôn ngữ: {task_lang} | Số phân cảnh: {len(scenes)}")
    print("-" * 70)

    active_task_path = IN_PROGRESS_DIR / f"task_{job_id}.json"
    if task_path.exists() and str(task_path) != str(active_task_path):
        try:
            shutil.move(str(task_path), str(active_task_path))
        except Exception:
            pass

    work_dir = Path(f"/tmp/job_{job_id}")
    work_dir.mkdir(parents=True, exist_ok=True)
    out_mp4 = COMPLETED_DIR / f"video_{job_id}.mp4"
    local_render_mp4 = work_dir / f"render_gpu_{job_id}.mp4"

    # 1. Sinh Voiceover
    print("   1️⃣ Đang sinh giọng đọc AI Edge-TTS...")
    audio_path = str(work_dir / "voice.mp3")
    ok_voice = await generate_voice_edge_tts(full_narration, voice, audio_path)
    if not ok_voice:
        print(f"❌ Thất bại khi tạo giọng đọc cho #{job_id}")
        return False

    # Lấy thời lượng âm thanh thực tế
    probe_res = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", audio_path],
        capture_output=True, text=True
    )
    total_duration = float(probe_res.stdout.strip() or 30.0)
    print(f"   ⏱️ Thời lượng audio: {total_duration:.1f} giây")

    # 2. Tạo phụ đề Karaoke Whisper ASS (Word-by-word)
    print(f"   2️⃣ Đang nhận diện timestamp từng từ bằng GPU Whisper ({'vi' if is_vi else 'en'})...")
    words = transcribe_with_whisper(audio_path, language=task_lang)
    sub_path = str(work_dir / "subs.ass")
    if words:
        create_dynamic_ass_subtitles(words, sub_path, total_duration=total_duration, is_vietnamese=is_vi)
        print("   ✨ Đã tạo file phụ đề ASS động kèm CTA Banner!")
    else:
        sub_path = None

    # 3. Chuẩn bị Scene Clips theo chỉ thị thông minh của VideoStrategistAgent
    print(f"   3️⃣ Đang xử lý {len(scenes)} phân cảnh video 9:16 (Áp dụng chiến lược VideoStrategist)...")
    scene_clips = []
    time_per_scene = total_duration / max(len(scenes), 1)

    for idx, sc in enumerate(scenes):
        s_id = sc.get("scene_id", idx + 1)
        s_clip = str(work_dir / f"scene_{s_id:03d}.mp4")
        q = sc.get("search_query") or sc.get("stock_query") or sc.get("visual_keyword") or title
        v_prompt = sc.get("ai_prompt") or sc.get("visual_prompt") or q
        dur_s = float(sc.get("estimated_seconds") or time_per_scene)
        search_and_download_scene_media(
            query=q,
            visual_prompt=v_prompt,
            out_path=s_clip,
            duration=dur_s,
            pexels_key=pexels_key,
            pixabay_key=pixabay_key,
            scene_id=s_id,
            scene_meta=sc
        )
        scene_clips.append(s_clip)

    # 4. Ghép Clips & Render hoàn chỉnh bằng GPU vào ổ đĩa nội bộ /tmp
    print(f"   4️⃣ Đang lắp ráp và kết xuất video bằng {ENCODER} GPU...")
    concat_txt = str(work_dir / "concat.txt")
    with open(concat_txt, "w", encoding="utf-8") as f_c:
        for sc_p in scene_clips:
            f_c.write(f"file '{sc_p}'\n")

    silent_mp4 = str(work_dir / "silent.mp4")
    subprocess.run([
        "ffmpeg", "-y",
        "-f", "concat", "-safe", "0", "-i", concat_txt,
        "-c:v", "copy",
        "-an",
        silent_mp4
    ], check=True)

    mux_args = [
        "ffmpeg", "-y",
        "-i", silent_mp4,
        "-i", audio_path,
    ]
    if sub_path and Path(sub_path).exists():
        mux_args.extend(["-vf", f"ass={sub_path}"])
        if ENCODER == "h264_nvenc":
            mux_args.extend([
                "-c:v", ENCODER,
                "-rc:v", "vbr", "-cq:v", "18",
                "-b:v", "12M", "-maxrate:v", "18M", "-bufsize:v", "24M",
                "-spatial_aq", "1", "-temporal_aq", "1",
                "-preset", "p5"
            ])
        else:
            mux_args.extend(["-c:v", ENCODER, "-crf", "18", "-preset", "medium"])
    else:
        mux_args.extend(["-c:v", "copy"])
    mux_args.extend([
        "-c:a", "aac",
        "-b:a", "256k",
        "-shortest",
        str(local_render_mp4)
    ])
    subprocess.run(mux_args, check=True)

    # 5. Lưu trữ video sang Google Drive an toàn
    if local_render_mp4.exists() and local_render_mp4.stat().st_size > 1000:
        print(f"   💾 Đang lưu video lên Google Drive ({local_render_mp4.stat().st_size / (1024*1024):.1f} MB)...\n")
        shutil.copy2(str(local_render_mp4), str(out_mp4))
        try:
            os.sync()
        except Exception:
            pass
    else:
        print(f"❌ File render bị lỗi hoặc rỗng cho #{job_id}")
        return False

    cost_time = time.time() - t_start
    meta = {
        "job_id": job_id,
        "title": title,
        "duration_seconds": total_duration,
        "render_time_seconds": round(cost_time, 1),
        "encoder": ENCODER,
        "completed_at": time.strftime('%Y-%m-%d %H:%M:%S'),
        "file_name": f"video_{job_id}.mp4"
    }
    with open(COMPLETED_DIR / f"meta_{job_id}.json", "w", encoding="utf-8") as f_m:
        json.dump(meta, f_m, ensure_ascii=False, indent=2)
    try:
        os.sync()
    except Exception:
        pass

    archive_file = ARCHIVE_DIR / f"task_{job_id}.json"
    if active_task_path.exists() and str(active_task_path) != str(archive_file):
        try:
            shutil.move(str(active_task_path), str(archive_file))
        except Exception:
            pass
    lock_file = LOCKS_DIR / f"task_{job_id}.lock"
    if lock_file.exists():
        try:
            lock_file.unlink(missing_ok=True)
        except Exception:
            pass
    shutil.rmtree(str(work_dir), ignore_errors=True)

    # 6. Bắn thông báo tức thì về Telegram từ Colab
    bot_token = task.get("telegram_bot_token")
    chat_ids = task.get("telegram_chat_ids", [])
    if bot_token and chat_ids:
        for cid in chat_ids:
            try:
                requests.post(
                    f"https://api.telegram.org/bot{bot_token}/sendMessage",
                    json={
                        "chat_id": cid,
                        "text": f"🎉 *[COLAB GPU FARM] VIDEO #{job_id} ĐÃ RENDER HOÀN TẤT!*\n\n📌 *Tiêu đề:* {title}\n⚡ *Thời gian:* {cost_time:.1f}s (NVENC GPU)\n☁️ Đã lưu an toàn lên Google Drive. Bot Telegram đang tải về gửi cho nhóm!",
                        "parse_mode": "Markdown"
                    },
                    timeout=5
                )
            except Exception:
                pass

    print("=" * 75)
    print(f"🎉 THÀNH CÔNG! VIDEO #{job_id} ĐÃ HOÀN TẤT TRONG {cost_time:.1f} GIÂY!")
    print(f"📁 File đã lưu vào Google Drive: {out_mp4}")
    print(f"🤖 Bot Telegram sẽ tự động quét thấy và gửi video cho bạn!")
    print("=" * 75)
    return True

# Chạy danh sách tác vụ
for t_path in active_tasks:
    try:
        await process_single_task(t_path)
    except Exception as ex_p:
        print(f'❌ Lỗi khi render {t_path.name}: {ex_p}')


In [ ]:
# [BƯỚC 7] TỰ ĐỘNG TẮT GPU & NGẮT KẾT NỐI (TIẾT KIỆM COMPUTE UNITS PRO+ TUYỆT ĐỐI)
# 🛡️ BẢO VỆ TÀI KHOẢN PRO+: Khi render xong (hoặc không có task), Colab sẽ tự động ngắt kết nối runtime ngay lập tức!
# Nhờ vậy bạn KHÔNG BỊ TRỪ COMPUTE UNITS khi treo máy, 1 tháng dùng tẹt ga không bao giờ lo hết quota!

import time
import os
import json

print("=" * 75)
print("🛡️ BẢO VỆ QUOTA: CẬP NHẬT TRẠNG THÁI & TỰ ĐỘNG NGẮT KẾT NỐI RUNTIME...")
print("✅ Toàn bộ video đã được lưu vào Google Drive an toàn.")
print("🤖 Bot Telegram đã/đang quét Drive và gửi video về nhóm cho bạn.")
print("⚡ Đang tắt GPU và unassign runtime ngay bây giờ (tiêu tốn 0 Compute Units khi nghỉ)...")
print("=" * 75)

try:
    if "update_cluster_heartbeat" in globals():
        update_cluster_heartbeat(status="COMPLETED")
except Exception as ex_hb:
    print(f"Lưu trạng thái: {ex_hb}")

time.sleep(3)
try:
    from google.colab import runtime
    runtime.unassign()
    print("🔒 Đã giải phóng GPU & ngắt kết nối thành công!")
except Exception as e:
    print(f"Hoàn thành phiên làm việc! (Lưu ý ngoài Colab: {e})")
